#  TMS - Reinforcement Learning Signal Controller Training

Train RL agents for traffic signal control:
- DQN (Deep Q-Network)
- Double DQN (Reduced overestimation)
- Dueling DQN (Value/Advantage streams)

Features:
-  **Environmental reward** (carbon reduction)
-  **Multi-intersection coordination**
-  **LSTM prediction integration**

---

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/tms2_colab_training'
MODEL_PATH = f'{DRIVE_PATH}/models'

import os
os.makedirs(f'{MODEL_PATH}/rl', exist_ok=True)

print(f"Working directory: {DRIVE_PATH}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/tms2_colab_training


In [ ]:
!pip install -q gymnasium torch numpy matplotlib tqdm
from models import DQNNetwork, DuelingDQNNetwork
from replay_buffer import ReplayBuffer
from env import TrafficSignalEnv

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import pickle


Using device: cuda
GPU: Tesla T4


## 2. Traffic Signal Environment

In [ ]:
class TrafficSignalEnv:
    def __init__(self, config=None):
        self.config = config or {}

        self.state_dim = 10
        self.action_dim = 2  # 0: keep, 1: change

        # Signal timing constraints
        self.min_green = 15
        self.max_green = 90
        self.max_steps = 500

        # Reward weights
        self.reward_weights = {
            'throughput': 0.4,
            'wait_time': 0.3,
            'emissions': 0.2,
            'coordination': 0.1
        }

        self.reset()

    def reset(self):
        self.current_step = 0
        self.current_phase = 0
        self.time_in_phase = 0
        self.total_wait = 0
        self.total_throughput = 0
        self.total_emissions = 0

        # Initialize queues
        self.queues = {
            'N': np.random.randint(5, 20),
            'S': np.random.randint(5, 20),
            'E': np.random.randint(5, 20),
            'W': np.random.randint(5, 20)
        }

        self.hour = np.random.randint(0, 24)
        self.is_peak = 1 if self.hour in [8, 9, 17, 18, 19] else 0

        return self._get_state()

    def _get_state(self):
        total = sum(self.queues.values())
        density = min(1.0, total / 100)
        avg_speed = max(5, 40 - total * 0.3)

        state = np.array([
            self.current_phase / 3.0,
            self.time_in_phase / self.max_green,
            self.queues['N'] / 50.0,
            self.queues['S'] / 50.0,
            self.queues['E'] / 50.0,
            self.queues['W'] / 50.0,
            density,
            avg_speed / 60.0,
            self.hour / 23.0,
            self.is_peak
        ], dtype=np.float32)

        return state

    def step(self, action):
        self.current_step += 1
        self.time_in_phase += 5  # 5 second time step

        # Phase change logic
        if action == 1 and self.time_in_phase >= self.min_green:
            self.current_phase = (self.current_phase + 1) % 4
            self.time_in_phase = 0
        elif self.time_in_phase >= self.max_green:
            self.current_phase = (self.current_phase + 1) % 4
            self.time_in_phase = 0

        # Process traffic
        throughput, wait_time, emissions = self._process_traffic()
        self._add_arrivals()

        # Calculate reward
        reward = self._calculate_reward(throughput, wait_time, emissions)

        # Update hour occasionally
        if self.current_step % 60 == 0:
            self.hour = (self.hour + 1) % 24
            self.is_peak = 1 if self.hour in [8, 9, 17, 18, 19] else 0

        done = self.current_step >= self.max_steps

        info = {
            'throughput': throughput,
            'wait_time': wait_time,
            'emissions': emissions
        }

        return self._get_state(), reward, done, info

    def _process_traffic(self):
        # Phase 0/2: N-S, Phase 1/3: E-W
        if self.current_phase in [0, 2]:
            active = ['N', 'S']
            waiting = ['E', 'W']
        else:
            active = ['E', 'W']
            waiting = ['N', 'S']

        throughput = 0
        for d in active:
            vehicles = min(self.queues[d], np.random.randint(2, 5))
            self.queues[d] = max(0, self.queues[d] - vehicles)
            throughput += vehicles

        wait_time = sum(self.queues[d] * 5 for d in waiting)
        emissions = sum(self.queues.values()) * 0.1

        self.total_throughput += throughput
        self.total_wait += wait_time
        self.total_emissions += emissions

        return throughput, wait_time, emissions

    def _add_arrivals(self):
        rate = 5 if self.is_peak else 2
        for d in self.queues:
            self.queues[d] = min(50, self.queues[d] + np.random.poisson(rate))

    def _calculate_reward(self, throughput, wait_time, emissions):
        w = self.reward_weights

        throughput_r = throughput / 10.0
        wait_r = -wait_time / 500.0
        emissions_r = -emissions / 50.0

        queue_balance = 1.0 - np.std(list(self.queues.values())) / 25.0

        reward = (
            w['throughput'] * throughput_r +
            w['wait_time'] * wait_r +
            w['emissions'] * emissions_r +
            w['coordination'] * queue_balance
        )

        return reward

## 3. DQN Agent

In [ ]:
# Experience replay buffer
Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


class DuelingDQNNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()

        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )

        # Value stream
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        # Advantage stream
        self.advantage = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        features = self.feature(x)
        value = self.value(features)
        advantage = self.advantage(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

In [ ]:
class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=0.0001,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=0.995,
        buffer_size=50000,
        batch_size=512,
        target_update=100,
        dueling=False,
        double_dqn=True
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.double_dqn = double_dqn
        self.dueling = dueling

        # Networks
        NetworkClass = DuelingDQNNetwork if dueling else DQNNetwork
        self.policy_net = NetworkClass(state_dim, action_dim).to(device)
        self.target_net = NetworkClass(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size)

        self.steps_done = 0

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax(1).item()

    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

    def update(self):
        if len(self.memory) < self.batch_size:
            return None

        transitions = self.memory.sample(self.batch_size)
        batch = Transition(*zip(*transitions))

        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).unsqueeze(1).to(device)

        # Current Q values
        current_q = self.policy_net(states).gather(1, actions)

        # Next Q values (Double DQN)
        with torch.no_grad():
            if self.double_dqn:
                next_actions = self.policy_net(next_states).argmax(1, keepdim=True)
                next_q = self.target_net(next_states).gather(1, next_actions)
            else:
                next_q = self.target_net(next_states).max(1, keepdim=True)[0]

            target_q = rewards + self.gamma * next_q * (1 - dones)

        # Loss
        loss = F.smooth_l1_loss(current_q, target_q)

        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        # Update target network
        self.steps_done += 1
        if self.steps_done % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        return loss.item()

    def save(self, path):
        torch.save({
            'policy_net': self.policy_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'epsilon': self.epsilon,
            'steps_done': self.steps_done,
            'config': {
                'state_dim': self.state_dim,
                'action_dim': self.action_dim,
                'gamma': self.gamma,
                'double_dqn': self.double_dqn,
                'dueling': self.dueling
            }
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=device)
        self.policy_net.load_state_dict(checkpoint['policy_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.optimizer.load_state_dict(checkpoint['optimizer'])
        self.epsilon = checkpoint['epsilon']
        self.steps_done = checkpoint['steps_done']

## 4. Training Loop

In [ ]:
def train_agent(
    agent,
    env,
    episodes=1000,
    save_path=None,
    save_frequency=100
):
    episode_rewards = []
    episode_throughputs = []
    episode_wait_times = []
    losses = []

    best_reward = float('-inf')

    for episode in tqdm(range(episodes), desc="Training"):
        state = env.reset()
        episode_reward = 0
        episode_loss = []

        done = False
        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_transition(state, action, reward, next_state, done)
            loss = agent.update()
            if loss is not None:
                episode_loss.append(loss)

            state = next_state
            episode_reward += reward

        # Record metrics
        episode_rewards.append(episode_reward)
        episode_throughputs.append(env.total_throughput)
        episode_wait_times.append(env.total_wait)
        if episode_loss:
            losses.append(np.mean(episode_loss))

        # Save best model
        if episode_reward > best_reward and save_path:
            best_reward = episode_reward
            agent.save(f"{save_path}_best.pt")

        # Periodic save
        if save_path and (episode + 1) % save_frequency == 0:
            agent.save(f"{save_path}_ep{episode+1}.pt")
            print(f"\nEpisode {episode+1}: Reward={episode_reward:.2f}, "
                  f"Throughput={env.total_throughput}, ε={agent.epsilon:.3f}")

    return {
        'rewards': episode_rewards,
        'throughputs': episode_throughputs,
        'wait_times': episode_wait_times,
        'losses': losses
    }

## 5. Train Multiple Agents

In [ ]:
EPISODES = 1000
SAVE_FREQ = 100
env = TrafficSignalEnv()

print(f"Environment: state_dim={env.state_dim}, action_dim={env.action_dim}")
print(f"Training for {EPISODES} episodes...")

results = {}

Environment: state_dim=10, action_dim=2
Training for 1000 episodes...


In [1]:
# 1. Standard DQN
agent_dqn = DQNAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    double_dqn=False,
    dueling=False
)

results['dqn'] = train_agent(
    agent_dqn, env, EPISODES,
    save_path=f"{MODEL_PATH}/rl/dqn",
    save_frequency=SAVE_FREQ
)

agent_dqn.save(f"{MODEL_PATH}/rl/dqn_final.pt")

NameError: name 'DQNAgent' is not defined

In [2]:
agent_ddqn = DQNAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    double_dqn=True,
    dueling=False
)
results['double_dqn'] = train_agent(
    agent_ddqn, env, EPISODES,
    save_path=f"{MODEL_PATH}/rl/double_dqn",
    save_frequency=SAVE_FREQ
)

agent_ddqn.save(f"{MODEL_PATH}/rl/double_dqn_final.pt")

NameError: name 'DQNAgent' is not defined

In [ ]:
agent_dueling = DQNAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    double_dqn=True,
    dueling=True
)

results['dueling_dqn'] = train_agent(
    agent_dueling, env, EPISODES,
    save_path=f"{MODEL_PATH}/rl/dueling_dqn",
    save_frequency=SAVE_FREQ
)

agent_dueling.save(f"{MODEL_PATH}/rl/dueling_dqn_final.pt")


Training Dueling DQN...


Training:   1%|▏         | 14/1000 [00:32<37:50,  2.30s/it]


KeyboardInterrupt: 

## 6. Compare Results

In [ ]:
# Smooth rewards for plotting
def smooth(data, window=50):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rewards
ax = axes[0, 0]
for name, data in results.items():
    ax.plot(smooth(data['rewards']), label=name.upper(), alpha=0.8)
ax.set_title('Episode Rewards (Smoothed)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# Throughput
ax = axes[0, 1]
for name, data in results.items():
    ax.plot(smooth(data['throughputs']), label=name.upper(), alpha=0.8)
ax.set_title('Episode Throughput (Smoothed)')
ax.set_xlabel('Episode')
ax.set_ylabel('Vehicles')
ax.legend()
ax.grid(True, alpha=0.3)

# Wait times
ax = axes[1, 0]
for name, data in results.items():
    ax.plot(smooth(data['wait_times']), label=name.upper(), alpha=0.8)
ax.set_title('Episode Wait Time (Smoothed)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Wait (sec)')
ax.legend()
ax.grid(True, alpha=0.3)

# Final comparison bar chart
ax = axes[1, 1]
names = list(results.keys())
final_rewards = [np.mean(results[n]['rewards'][-100:]) for n in names]
colors = ['#4CAF50', '#2196F3', '#FF9800']
bars = ax.bar(names, final_rewards, color=colors, edgecolor='black')
ax.set_title('Final Average Reward (Last 100 Episodes)')
ax.set_ylabel('Average Reward')
for bar, val in zip(bars, final_rewards):
    ax.annotate(f'{val:.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom')

plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/rl/training_comparison.png', dpi=150)
plt.show()

In [ ]:
import pandas as pd

In [ ]:
summary = {}
for name, data in results.items():
    summary[name] = {
        'final_reward': np.mean(data['rewards'][-100:]),
        'final_throughput': np.mean(data['throughputs'][-100:]),
        'final_wait_time': np.mean(data['wait_times'][-100:]),
        'max_reward': max(data['rewards'])
    }

summary_df = pd.DataFrame(summary).T
summary_df = summary_df.sort_values('final_reward', ascending=False)
print(summary_df)

best_agent = summary_df.index[0]
print(f"\n🏆 Best Agent: {best_agent.upper()}")

## 7. Save Final Models

In [ ]:
training_summary = {
    'training_date': pd.Timestamp.now().isoformat(),
    'episodes': EPISODES,
    'environment': {
        'state_dim': env.state_dim,
        'action_dim': env.action_dim,
        'reward_weights': env.reward_weights
    },
    'results': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in summary.items()},
    'best_agent': best_agent
}

with open(f'{MODEL_PATH}/rl/training_summary.json', 'w') as f:
    json.dump(training_summary, f, indent=2)

# Save best agent for deployment
import shutil
best_model_path = f"{MODEL_PATH}/rl/{best_agent}_final.pt"
shutil.copy(best_model_path, f"{MODEL_PATH}/rl_signal_controller.pt")
